In [9]:
import fitz 
import re

In [6]:
doc=fitz.open("../data/raw/158-vbhn-vpqh.pdf")
len(doc)

133

In [8]:
for i in range(1,3):
    page=doc.load_page(i)
    text=page.get_text()
    print(text)


2 
hàng; việc xử lý nợ xấu, tài sản bảo đảm của khoản nợ xấu của tổ chức tín dụng, 
chi nhánh ngân hàng nước ngoài, tổ chức mà Nhà nước sở hữu 100% vốn điều lệ 
có chức năng mua, bán, xử lý nợ. 
Điều 2. Đối tượng áp dụng 
1. Tổ chức tín dụng. 
2. Chi nhánh ngân hàng nước ngoài. 
3. Văn phòng đại diện tại Việt Nam của tổ chức tín dụng nước ngoài, tổ 
chức nước ngoài khác có hoạt động ngân hàng (sau đây gọi là văn phòng đại 
diện nước ngoài). 
4. Tổ chức mà Nhà nước sở hữu 100% vốn điều lệ có chức năng mua, bán, 
xử lý nợ (sau đây gọi là tổ chức mua bán, xử lý nợ). 
5. Cơ quan, tổ chức, cá nhân có liên quan đến việc thành lập, tổ chức, hoạt 
động, can thiệp sớm, kiểm soát đặc biệt, tổ chức lại, giải thể, phá sản tổ chức tín 
dụng; việc thành lập, tổ chức, hoạt động, can thiệp sớm, giải thể, chấm dứt hoạt 
động của chi nhánh ngân hàng nước ngoài; việc thành lập, hoạt động của văn 
phòng đại diện nước ngoài; việc xử lý nợ xấu, tài sản bảo đảm của khoản nợ xấu 
của tổ chức tín dụng, chi nhá

In [12]:
def extract_and_clean_pdf(pdf_path, num_pages=5):
    doc = fitz.open(pdf_path)
    extracted_text = ""
    
    for page_num in range(1, min(num_pages, doc.page_count)):
        page = doc.load_page(page_num)
        
        # 1. CHIẾN THUẬT CẮT LỀ (Bounding Box)
        # Lấy kích thước của trang giấy
        page_width = page.rect.width
        page_height = page.rect.height
        
        # Cắt bỏ 50 pixel lề trên và 50 pixel lề dưới (nơi thường chứa số trang)
        # Tọa độ: (x0, y0, x1, y1)
        clip_rect = fitz.Rect(0, 50, page_width, page_height - 50)
        
        # Chỉ lấy text trong vùng đã cắt
        text = page.get_text("text", clip=clip_rect)
        
        # 2. LÀM SẠCH NHẸ NHÀNG (Chỉ nối câu bị đứt, TUYỆT ĐỐI KHÔNG XÓA SỐ)
        # Đổi các dấu xuống dòng (\n) thành khoảng trắng, trừ khi trước nó là dấu kết thúc câu
        text = re.sub(r'(?<![.:;])\n', ' ', text)
        
        # Xóa khoảng trắng thừa
        text = re.sub(r'\s+', ' ', text).strip()
        
        extracted_text += text + "\n\n"
        
    return extracted_text

# Chạy thử code mới
pdf_file = "../data/raw/158-vbhn-vpqh.pdf" # Đổi tên file cho đúng
clean_data = extract_and_clean_pdf(pdf_file)
print(clean_data)

hàng; việc xử lý nợ xấu, tài sản bảo đảm của khoản nợ xấu của tổ chức tín dụng, chi nhánh ngân hàng nước ngoài, tổ chức mà Nhà nước sở hữu 100% vốn điều lệ có chức năng mua, bán, xử lý nợ. Điều 2. Đối tượng áp dụng 1. Tổ chức tín dụng. 2. Chi nhánh ngân hàng nước ngoài. 3. Văn phòng đại diện tại Việt Nam của tổ chức tín dụng nước ngoài, tổ chức nước ngoài khác có hoạt động ngân hàng (sau đây gọi là văn phòng đại diện nước ngoài). 4. Tổ chức mà Nhà nước sở hữu 100% vốn điều lệ có chức năng mua, bán, xử lý nợ (sau đây gọi là tổ chức mua bán, xử lý nợ). 5. Cơ quan, tổ chức, cá nhân có liên quan đến việc thành lập, tổ chức, hoạt động, can thiệp sớm, kiểm soát đặc biệt, tổ chức lại, giải thể, phá sản tổ chức tín dụng; việc thành lập, tổ chức, hoạt động, can thiệp sớm, giải thể, chấm dứt hoạt động của chi nhánh ngân hàng nước ngoài; việc thành lập, hoạt động của văn phòng đại diện nước ngoài; việc xử lý nợ xấu, tài sản bảo đảm của khoản nợ xấu của tổ chức tín dụng, chi nhánh ngân hàng nước n

In [15]:
import fitz
import re

def extract_clean_and_chunk(pdf_path, num_pages=20):
    doc = fitz.open(pdf_path)
    full_text = ""
    
    # BƯỚC 1: ĐỌC VÀ LÀM SẠCH (Data Ingestion & Cleaning)
    for page_num in range(1, min(num_pages, doc.page_count)):
        page = doc.load_page(page_num)
        
        # Cắt lề trên dưới 50px để loại bỏ số trang và tiêu đề lặp
        clip_rect = fitz.Rect(0, 50, page.rect.width, page.rect.height - 50)
        text = page.get_text("text", clip=clip_rect)
        
        # Xóa dấu ngắt dòng vô lý, giữ lại các dấu chấm câu
        text = re.sub(r'(?<![.:;])\n', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        
        full_text += text + "\n\n"
        
    # BƯỚC 2: BĂM NHỎ DỮ LIỆU (Semantic Chunking)
    # Dùng Regex Lookahead để cắt ngay trước chữ "Điều [số]."
    # \b đảm bảo nó là từ độc lập, \s+ bắt mọi khoảng trắng
    chunks = re.split(r'(?=\bĐiều\s+\d+\.)', full_text)
    
    # Lọc bỏ các đoạn rác quá ngắn (dưới 50 ký tự)
    valid_chunks = [c.strip() for c in chunks if len(c.strip()) > 50]
    
    return valid_chunks

# CHẠY THỬ LUỒNG HOÀN CHỈNH
pdf_file = "../data/raw/158-vbhn-vpqh.pdf" # Đảm bảo đường dẫn chuẩn
law_chunks = extract_clean_and_chunk(pdf_file, num_pages=20)

print(f"🎉 TỔNG SỐ 'ĐIỀU' ĐÃ CẮT ĐƯỢC: {len(law_chunks)} đoạn\n")

# In thử 3 đoạn đầu tiên xem đã "đẹp trai" chưa
for i, chunk in enumerate(law_chunks[:3]):
    print(f"--- CHUNK {i+1} (Độ dài: {len(chunk)} ký tự) ---")
    print(chunk[:250] + "...\n")

🎉 TỔNG SỐ 'ĐIỀU' ĐÃ CẮT ĐƯỢC: 39 đoạn

--- CHUNK 1 (Độ dài: 188 ký tự) ---
hàng; việc xử lý nợ xấu, tài sản bảo đảm của khoản nợ xấu của tổ chức tín dụng, chi nhánh ngân hàng nước ngoài, tổ chức mà Nhà nước sở hữu 100% vốn điều lệ có chức năng mua, bán, xử lý nợ....

--- CHUNK 2 (Độ dài: 843 ký tự) ---
Điều 2. Đối tượng áp dụng 1. Tổ chức tín dụng. 2. Chi nhánh ngân hàng nước ngoài. 3. Văn phòng đại diện tại Việt Nam của tổ chức tín dụng nước ngoài, tổ chức nước ngoài khác có hoạt động ngân hàng (sau đây gọi là văn phòng đại diện nước ngoài). 4. Tổ...

--- CHUNK 3 (Độ dài: 298 ký tự) ---
Điều 3. Áp dụng tập quán thương mại Tổ chức, cá nhân tham gia hoạt động ngân hàng được quyền thỏa thuận áp dụng tập quán thương mại sau đây: 1. Tập quán thương mại quốc tế do Phòng Thương mại quốc tế ban hành; 2. Tập quán thương mại khác không trái v...

